# Plan Performance Analysis

Analyze a **completed** `terraform plan` from **`TF_LOG=json`** trace output (for example via `TF_LOG_PATH`).

Parses graph refresh lines, planned changes, and `PlanResourceChange` RPC activity from provider/core diagnostic JSON.

Capture example:

```bash
export TF_LOG=json
export TF_LOG_PATH=plan-tflog.log
terraform plan
export TERRAFORM_LOG_PATH=plan-tflog.log
```

For **`terraform plan -json`** UI output, use `sdk-plan/plan-analysis.ipynb`. For hung or partial plans use `plan/hang-analysis.ipynb`. Run `whatisit.ipynb` if unsure.

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd
import commonlib.prep_plan_log_data as prep_plan_log_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg


In [ ]:
c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)
normalized_records = prep_plan_log_data.load_normalized_records()
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No TF_LOG plan trace records found. Capture with TF_LOG=json during plan "
        "or use sdk-plan/plan-analysis.ipynb for terraform plan -json UI output."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

In [ ]:
starts = (
    df[df["type"] == "refresh_start"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
starts["refresh_start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource_id").cumcount() + 1
starts = starts[
    ["resource_id", "run", "refresh_start_timestamp", "resource", "resource_type", "resource_name"]
]

ends = (
    df[df["type"] == "refresh_complete"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
ends["refresh_complete_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource_id").cumcount() + 1
ends = ends[["resource_id", "run", "refresh_complete_timestamp"]]

df_merged_refresh = starts.merge(ends, on=["resource_id", "run"], how="inner")
df_merged_refresh["time_diff_minutes"] = (
    (df_merged_refresh["refresh_complete_timestamp"] - df_merged_refresh["refresh_start_timestamp"])
    .dt.total_seconds()
    / 60
)

df_planned_change = df[df["type"] == "planned_change"]

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "refresh_start", top_n=10)
gencharts.generate_plt_by_resource_type(df, "refresh_complete", top_n=10)
gencharts.generate_plt_by_resource_type(df, "planned_change", top_n=10)

## Duration Analysis

In [ ]:
if not df_merged_refresh.empty:
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="average", top_n=10)
else:
    print("No matched refresh_start/refresh_complete pairs to chart.")

## Longest Refresh Times

In [ ]:
if df_merged_refresh.empty:
    print("No matched refresh_start/refresh_complete pairs to list.")
else:
    display(
        df_merged_refresh[
            ["resource", "resource_type", "refresh_start_timestamp", "refresh_complete_timestamp", "time_diff_minutes", "run"]
        ]
        .copy()
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )

## Provider API activity

SDK DEBUG activity from the log for **completed** runs: **when** traffic spiked (timeline) and key endpoint averages. Retry/404 detail is in the exported report only. For raw HTTP pairs use `sdk-plan/sdk-analysis.ipynb`.


In [ ]:
import commonlib.prep_hang_data as hang

TAIL_MINUTES = 5
WORKFLOW = "plan"
classification, _, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
summary = hang.hang_summary_for_workflow(
    counters, TAIL_MINUTES, WORKFLOW, classification
)
df_sdk_timeline, sdk_timeline_summary, df_sdk_rates = hang.display_provider_api_activity(
    counters, summary
)
hang.display_issue_attribution(hang.build_issue_attribution(counters, summary, WORKFLOW))


## Export report

Writes `{capture-stem}-report.json` next to the log (full SDK tables including `sdk_timeline`, retry, and 404). See **`HOW-TO-READ-RESULTS.md`**.


In [ ]:
import commonlib.run_report as run_report

run_report.write_run_report(
    c.TERRAFORM_LOG_PATH,
    WORKFLOW,
    {
        **run_report.issue_attribution_bundle(counters, summary, WORKFLOW),
        "completed_refreshes": run_report.dataframe_records(df_merged_refresh),
        **run_report.sdk_report_sections(
            counters, duration_minutes=summary.get("duration_minutes")
        ),
    },
)
